Objective

Convert the Behavioral Intelligence features into:

Investor Personas
Confidence Score
Next Best Action (NBA)
Rule Explanations

This is the heart of NudgeIQ.

behavioral_intelligence.csv
        │
        ▼
Load Data
        │
        ▼
Apply Rule Engine
        │
        ▼
Assign Persona
        │
        ▼
Assign Next Best Action
        │
        ▼
Calculate Confidence
        │
        ▼
rule_engine_output.csv

In [1]:
import pandas as pd
import numpy as np

# Load behavioural intelligence dataset
df = pd.read_csv("../data/processed/behavioral_intelligence.csv")

print(df.shape)
df.head()

(11162, 39)


,age,job,marital,education,default,balance,housing,loan,contact,day,...,Digital_Engagement,Risk_Appetite,Loyalty_Potential,duration_z,Responsiveness,Investment_Experience,Financial_Index,Engagement_Index,Investor_Psychology_Index,Behavioral_Intelligence_Score
0,59,admin.,married,secondary,no,2343,yes,no,unknown,5,...,-1.264371,-0.580787,-0.004129,1.930226,1.960333,0.838930,0.106749,-0.004264,0.336171,0.217928
1,56,admin.,married,secondary,no,45,no,no,unknown,5,...,-1.264371,-0.788295,-0.128098,3.154612,2.933025,0.499468,-0.258376,0.284113,-0.126506,-0.057041
2,41,technician,married,secondary,no,1270,yes,no,unknown,5,...,-1.264371,-0.292982,-0.112901,2.929901,2.754507,-0.291969,-0.213617,0.227370,-0.389833,-0.165340
3,55,services,married,secondary,no,2476,yes,no,unknown,5,...,-1.264371,-0.425941,0.009354,0.596366,0.900671,0.641332,0.146459,-0.379844,0.296392,0.067199
4,54,admin.,married,tertiary,no,184,no,no,unknown,5,...,-1.387698,-0.055712,-0.114007,0.867171,0.999063,0.411153,-0.216875,-0.477867,0.503069,-0.089095


In [2]:
required_columns = [
    "Financial_Index",
    "Engagement_Index",
    "Investor_Psychology_Index",
    "Behavioral_Intelligence_Score"
]

missing = [c for c in required_columns if c not in df.columns]

if missing:
    raise ValueError(f"Missing columns: {missing}")

print("✅ All required columns available.")

✅ All required columns available.


In [3]:
def classify(score):
    if score < -0.5:
        return "Low"
    elif score <= 0.5:
        return "Medium"
    else:
        return "High"

df["Financial_Level"] = df["Financial_Index"].apply(classify)
df["Engagement_Level"] = df["Engagement_Index"].apply(classify)
df["Psychology_Level"] = df["Investor_Psychology_Index"].apply(classify)

In [4]:
def assign_persona(row):

    F = row["Financial_Level"]
    E = row["Engagement_Level"]
    P = row["Psychology_Level"]

    if F=="High" and E=="High" and P=="High":
        return "Premium Investor"

    elif F=="High" and E=="High":
        return "Growth Investor"

    elif F=="High" and E=="Low":
        return "Dormant Wealth Holder"

    elif F=="Medium" and E=="High":
        return "Emerging Investor"

    elif F=="Medium" and E=="Medium":
        return "Balanced Investor"

    elif F=="Low" and E=="High":
        return "Potential Investor"

    elif F=="Low" and E=="Low":
        return "Low Engagement User"

    else:
        return "General Investor"

In [5]:
df["Persona"] = df.apply(assign_persona, axis=1)

In [6]:
nba = {

"Premium Investor":
"Offer premium gold investment plans",

"Growth Investor":
"Recommend SIP increase",

"Dormant Wealth Holder":
"Send re-engagement campaign",

"Emerging Investor":
"Educational investment content",

"Balanced Investor":
"Portfolio diversification tips",

"Potential Investor":
"First investment incentives",

"Low Engagement User":
"Awareness campaign",

"General Investor":
"Regular engagement"
}

df["Next_Best_Action"] = df["Persona"].map(nba)

In [7]:
confidence = (
    df["Behavioral_Intelligence_Score"].abs() * 25
    + 50
)

df["Confidence"] = confidence.clip(50,100).round(1)

In [8]:
df["Explanation"] = (
    "Financial="
    + df["Financial_Level"]
    + ", Engagement="
    + df["Engagement_Level"]
    + ", Psychology="
    + df["Psychology_Level"]
)

In [9]:
output_path = "../data/processed/rule_engine_output.csv"

df.to_csv(output_path, index=False)

print("✅ Rule Engine completed.")
print(df[[
    "Persona",
    "Next_Best_Action",
    "Confidence"
]].head())

✅ Rule Engine completed.
             Persona                Next_Best_Action  Confidence
0  Balanced Investor  Portfolio diversification tips        55.4
1  Balanced Investor  Portfolio diversification tips        51.4
2  Balanced Investor  Portfolio diversification tips        54.1
3  Balanced Investor  Portfolio diversification tips        51.7
4  Balanced Investor  Portfolio diversification tips        52.2


In [10]:
print(df["Financial_Level"].value_counts())

Financial_Level
Medium    6901
Low       2635
High      1626
Name: count, dtype: int64


In [11]:
print(df["Engagement_Level"].value_counts())

Engagement_Level
Medium    4915
High      3144
Low       3103
Name: count, dtype: int64


In [12]:
print(df["Psychology_Level"].value_counts())

Psychology_Level
Medium    4994
Low       3705
High      2463
Name: count, dtype: int64


In [13]:
print(df["Persona"].value_counts())

Persona
General Investor         3427
Balanced Investor        3241
Emerging Investor        1907
Low Engagement User      1092
Premium Investor          707
Potential Investor        488
Dormant Wealth Holder     258
Growth Investor            42
Name: count, dtype: int64
